# Population-model training analysis

Reads the output of `ProVoice.training_scripts.run_population_pipeline` — four sweeps,
each in its own directory:

| section | stage | loss | window | grid |
|---|---|---|---|---|
| 1 | `corn_w10` | soft-CORN | 10 s | full (3 dropout x 2 lr) |
| 2 | `ce_w10`   | softmax/CE | 10 s | full |
| 3 | `corn_w5`  | soft-CORN | 5 s  | winner of stage 1 |
| 4 | `corn_w20` | soft-CORN | 20 s | winner of stage 1 |

Every section produces the same seven things, so the stages stay comparable:

1. **Per-configuration** table — set-MAE / set-accuracy averaged over folds and seeds.
2. **Per-configuration x validation-fold** table — the same, broken out by fold.
3. **Early-win table** — how often an epoch below `--min-select-epoch` would have won.
4. **Init vs best vs constant** — does training achieve anything, and does it beat a constant?
5. **Variance decomposition** — can this sweep resolve configurations at all?
6. **Paired configuration comparison** — config deltas with fold difficulty cancelled.
7. **Histograms and validation curves.**

Stages with a single configuration (3 and 4) use the identical format, just with one row.

**Two reference lines recur and both matter:**

* the **constant floor** — best set-MAE reachable by always predicting one fixed LoA
  (~1.32 globally on this cohort). A model above it is losing to "always say 1".
* the **untrained init** — the model before a single gradient step. Beating *this* is
  "training did something"; beating the *floor* is "what it did generalizes". Those are
  different claims and the tables below keep them apart.

In [ ]:
import sys, pathlib, warnings, itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# The notebook lives in src/ProVoice/training_analysis/; walk up to find src/.
_SRC = pathlib.Path.cwd()
for _ in range(6):
    if (_SRC / "ProVoice").is_dir():
        break
    _SRC = _SRC.parent
sys.path.insert(0, str(_SRC))

# Imported, not re-implemented: anything recomputed from the curves must use the
# SAME smoothing the sweep used, or it answers a different question.
from ProVoice.training_scripts.sweep_population_hparams import smooth, SMOOTH_WINDOW

%matplotlib inline
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------- configuration
BASE = pathlib.Path("../../../results/pop_pipeline")   # the pipeline's --outdir
MIN_SELECT_EPOCH = 3      # must match the value the sweep ran with
LOA = [0, 1, 2, 3, 4]

STAGES = [
    ("corn_w10", "1. Full grid — soft-CORN, 10 s window"),
    ("ce_w10",   "2. Softmax / cross-entropy ablation — 10 s window"),
    ("corn_w5",  "3. 5 s window — soft-CORN, winning configuration"),
    ("corn_w20", "4. 20 s window — soft-CORN, winning configuration"),
]
print(f"BASE = {BASE.resolve()}")
for name, _ in STAGES:
    p = BASE / name / "sweep_results.csv"
    n = (sum(1 for _ in p.open()) - 1) if p.exists() else 0
    print(f"  {name:<10} {'found' if p.exists() else 'MISSING':<8} {n:>4} rows")

In [ ]:
# ------------------------------------------------------------------- loading

def load_stage(name: str) -> pd.DataFrame:
    """One stage's sweep_results.csv, or an empty frame if it has not run."""
    p = BASE / name / "sweep_results.csv"
    if not p.exists():
        return pd.DataFrame()
    df = pd.read_csv(p)
    # window/loss are constant within a stage, so the configuration IS (dropout, lr).
    df["config"] = ("drop=" + df["dropout"].astype(str)
                    + " lr=" + df["lr"].map(lambda x: f"{x:g}"))
    return df


def load_curves(name: str) -> pd.DataFrame:
    """Every per-epoch curve for a stage, tagged by its filename."""
    d = BASE / name / "runs"
    if not d.exists():
        return pd.DataFrame()
    out = []
    for f in sorted(d.glob("metrics_*.csv")):
        try:
            c = pd.read_csv(f)
        except Exception:
            continue
        if c.empty:
            continue
        c["tag"] = f.stem[len("metrics_"):]
        out.append(c)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def _config_from_tag(tag: str):
    """(config label, dropout, lr, fold) from a run tag `d0.15_lr0.002_f001-005_s0_w10_corn`."""
    try:
        drop = float(tag.split("_lr")[0][1:])
        rest = tag.split("_lr")[1]
        lr = float(rest.split("_f")[0])
        fold = rest.split("_f")[1].split("_s")[0].replace("-", "|")
        return f"drop={drop} lr={lr:g}", drop, lr, fold
    except Exception:
        return "?", np.nan, np.nan, "?"


def per_run_curves(name: str, min_select_epoch: int = MIN_SELECT_EPOCH) -> pd.DataFrame:
    """One row per RUN, derived from its retained per-epoch curve.

    The shared source for the early-win and init-vs-best tables — both need the
    curve, not the summary CSV, because the summary only records the FLOORED
    selection and never records the untrained init.
    """
    curves = load_curves(name)
    if curves.empty:
        return pd.DataFrame()
    rows = []
    for tag, c in curves.groupby("tag"):
        c = c.sort_values("epoch")
        mae = c["set_mae"].to_numpy(dtype=float)
        if len(mae) == 0:
            continue
        sm = smooth(mae, SMOOTH_WINDOW)
        j_free = int(np.argmin(sm))                        # no floor
        lo = min(int(min_select_epoch), len(sm) - 1)
        j_sel = lo + int(np.argmin(sm[lo:]))               # as the sweep selects
        cfg, drop, lr, fold = _config_from_tag(tag)

        def col(nm, default=np.nan):
            return float(c[nm].iloc[0]) if nm in c.columns and pd.notna(c[nm].iloc[0]) else default

        rows.append({
            "tag": tag, "config": cfg, "dropout": drop, "lr": lr, "val_pids": fold,
            "epochs_run": len(mae),
            "epoch_unfloored": int(c["epoch"].iloc[j_free]),
            "epoch_selected": int(c["epoch"].iloc[j_sel]),
            "mae_unfloored": float(sm[j_free]),
            "mae_selected": float(sm[j_sel]),
            "acc_selected": float(c["set_acc"].iloc[j_sel]),
            "init_mae": col("init_set_mae"), "init_acc": col("init_set_acc"),
            "const_mae": col("const_set_mae"), "const_acc": col("const_set_acc"),
        })
    return pd.DataFrame(rows)


def _fmt(df: pd.DataFrame, cols) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c in df:
            df[c] = df[c].astype(float).round(3)
    return df

In [ ]:
# ---------------------------------------------------------- tables 1 and 2

def table_per_config(df: pd.DataFrame) -> pd.DataFrame:
    """set-MAE / set-accuracy per configuration, averaged over folds and seeds.

    `set_mae` is `smoothed_best_set_mae` — what the sweep RANKS on — not the raw
    per-epoch minimum, which is optimistically biased by minimising over ~100
    noisy epochs. `mae_raw` keeps that minimum alongside so the gap is visible.

    `vs_const` is the number to read: negative beats a constant prediction,
    positive does not.
    """
    if df.empty:
        return pd.DataFrame()
    g = df.groupby("config", as_index=False).agg(
        n_runs=("seed", "size"),
        set_mae=("smoothed_best_set_mae", "mean"),
        mae_sd=("smoothed_best_set_mae", "std"),
        mae_raw=("best_set_mae", "mean"),
        set_acc=("set_acc_at_best", "mean"),
        acc_sd=("set_acc_at_best", "std"),
        qwk=("qwk_at_best", "mean"),
        const_mae=("const_set_mae", "mean"),
        const_acc=("const_set_acc", "mean"),
        E_star=("best_epoch_1se", "median"),
    )
    g["mae_se"] = g["mae_sd"] / np.sqrt(g["n_runs"].clip(lower=1))
    g["vs_const"] = g["set_mae"] - g["const_mae"]
    g["beats_const"] = g["vs_const"] < 0
    cols = ["config", "n_runs", "set_mae", "mae_sd", "mae_se", "mae_raw",
            "set_acc", "acc_sd", "qwk", "const_mae", "const_acc",
            "vs_const", "beats_const", "E_star"]
    return _fmt(g[cols].sort_values("set_mae").reset_index(drop=True),
                ["set_mae", "mae_sd", "mae_se", "mae_raw", "set_acc", "acc_sd",
                 "qwk", "const_mae", "const_acc", "vs_const"])


def table_per_config_fold(df: pd.DataFrame) -> pd.DataFrame:
    """The same metrics broken out by configuration x validation fold.

    Where fold heterogeneity becomes visible: constant floors range ~0.54 to ~1.47
    across folds, so a configuration's mean hides which folds it actually failed on.
    """
    if df.empty:
        return pd.DataFrame()
    g = df.groupby(["config", "val_pids"], as_index=False).agg(
        n_seeds=("seed", "size"),
        set_mae=("smoothed_best_set_mae", "mean"),
        mae_sd=("smoothed_best_set_mae", "std"),
        set_acc=("set_acc_at_best", "mean"),
        acc_sd=("set_acc_at_best", "std"),
        const_mae=("const_set_mae", "mean"),
        const_acc=("const_set_acc", "mean"),
        val_n=("val_n", "mean"),
        E_star=("best_epoch_1se", "median"),
    )
    g["vs_const"] = g["set_mae"] - g["const_mae"]
    g["beats_const"] = g["vs_const"] < 0
    return _fmt(g.sort_values(["config", "val_pids"]).reset_index(drop=True),
                ["set_mae", "mae_sd", "set_acc", "acc_sd", "const_mae",
                 "const_acc", "vs_const"])

In [ ]:
# ------------------------------ table 3: early wins, table 4: init vs best

def table_early_wins(pr: pd.DataFrame,
                     min_select_epoch: int = MIN_SELECT_EPOCH) -> pd.DataFrame:
    """How often an epoch BELOW the floor would have won, and what the floor cost.

    `--min-select-epoch` blocks early epochs from being checkpointed: epochs
    inside the LR warmup are not comparable to full-LR ones, and an unconstrained
    argmin of epoch 0 makes E*=0 — which would make the LODO runner train for
    zero epochs and ship random inits as population models.

    This says whether the floor is PROTECTING or BINDING:

    * `pct_early` — share of runs whose unfloored smoothed argmin was < the floor.
      High means held-out performance peaks before the model has trained, i.e.
      training hurts transfer from the start.
    * `mae_cost` — mean (selected - unfloored) over those runs: what the floor
      gave up. Compare against `mae_se` in table 1; below ~1 SE it is noise.
    """
    if pr.empty:
        return pd.DataFrame()
    pr = pr.copy()
    pr["early"] = pr["epoch_unfloored"] < min_select_epoch
    pr["cost"] = pr["mae_selected"] - pr["mae_unfloored"]
    g = pr.groupby("config", as_index=False).agg(
        n_runs=("early", "size"), n_early=("early", "sum"),
        median_epoch_unfloored=("epoch_unfloored", "median"),
        median_epoch_selected=("epoch_selected", "median"))
    cost = (pr[pr["early"]].groupby("config", as_index=False)["cost"]
            .mean().rename(columns={"cost": "mae_cost"}))
    g = g.merge(cost, on="config", how="left")
    g["pct_early"] = (100 * g["n_early"] / g["n_runs"]).round(1)
    g = g[["config", "n_runs", "n_early", "pct_early", "mae_cost",
           "median_epoch_unfloored", "median_epoch_selected"]]
    return _fmt(g.sort_values("pct_early", ascending=False).reset_index(drop=True),
                ["mae_cost"])


def table_init_vs_best(pr: pd.DataFrame) -> pd.DataFrame:
    """Untrained init -> selected model -> constant floor, per configuration.

    Separates two diagnoses the summary tables conflate:

    * `gain_vs_init` = selected - init. NEGATIVE means training achieved
      something. If it is ~0, the model learned nothing and every other number
      here is describing a random initialization.
    * `vs_const` = selected - constant floor. NEGATIVE means what it learned
      generalizes to held-out drivers.

    The interesting case on this cohort is `gain_vs_init < 0 < vs_const`:
    training works and still loses to "always predict one level", because driver
    preference does not transfer. `verdict` names which case each row is in.
    """
    if pr.empty or pr["init_mae"].isna().all():
        return pd.DataFrame()
    g = pr.groupby("config", as_index=False).agg(
        n_runs=("tag", "size"),
        init_mae=("init_mae", "mean"), init_acc=("init_acc", "mean"),
        best_mae=("mae_selected", "mean"), best_acc=("acc_selected", "mean"),
        const_mae=("const_mae", "mean"), const_acc=("const_acc", "mean"))
    g["gain_vs_init"] = g["best_mae"] - g["init_mae"]
    g["vs_const"] = g["best_mae"] - g["const_mae"]

    def verdict(r):
        learned, generalizes = r["gain_vs_init"] < -0.02, r["vs_const"] < 0
        if learned and generalizes:
            return "learns AND beats constant"
        if learned:
            return "learns, does NOT transfer"
        if generalizes:
            return "beats constant without learning (?)"
        return "no learning, no transfer"

    g["verdict"] = g.apply(verdict, axis=1)
    cols = ["config", "n_runs", "init_mae", "best_mae", "const_mae",
            "gain_vs_init", "vs_const", "init_acc", "best_acc", "const_acc", "verdict"]
    return _fmt(g[cols].sort_values("best_mae").reset_index(drop=True),
                [c for c in cols if c not in ("config", "n_runs", "verdict")])

In [ ]:
# ---------------- table 5: variance decomposition, table 6: paired comparison

def table_variance(df: pd.DataFrame) -> pd.DataFrame:
    """Is this sweep able to resolve configurations at all?

    A go/no-go check, not a description. Three sources of spread in
    `smoothed_best_set_mae`:

    * **fold** — SD of the fold means. Expected to dominate: folds differ in
      intrinsic difficulty (constant floors ~0.54 to ~1.47), and that is a
      property of the drivers, not of any configuration.
    * **seed** — mean within-(config, fold) SD across seeds. The noise floor of a
      single run.
    * **config** — SD of the configuration means, and their max-min spread. This
      is the signal the sweep exists to measure.

    The verdict compares the configuration SPREAD against the standard error of
    a configuration mean (seed SD / sqrt(runs per config)). If the spread is
    below ~2 SE the ranking is not resolvable, and picking a winner is picking
    noise — in which case take the most regularized configuration (the sweep's
    own tie-break) and spend the compute elsewhere.
    """
    if df.empty:
        return pd.DataFrame()
    m = "smoothed_best_set_mae"
    n_cfg = df["config"].nunique()
    per_cfg = df.groupby("config")[m].mean()
    fold_sd = float(df.groupby("val_pids")[m].mean().std(ddof=1)) if df["val_pids"].nunique() > 1 else np.nan
    seed_sd = float(df.groupby(["config", "val_pids"])[m].std(ddof=1).mean())
    cfg_sd = float(per_cfg.std(ddof=1)) if n_cfg > 1 else np.nan
    cfg_spread = float(per_cfg.max() - per_cfg.min()) if n_cfg > 1 else np.nan
    n_per_cfg = float(df.groupby("config").size().mean())
    cfg_se = seed_sd / np.sqrt(max(n_per_cfg, 1.0))
    if n_cfg <= 1:
        verdict = "single configuration — nothing to resolve"
    elif not np.isfinite(cfg_spread) or not np.isfinite(cfg_se) or cfg_se == 0:
        verdict = "insufficient data"
    elif cfg_spread > 2 * cfg_se:
        verdict = f"RESOLVABLE (spread {cfg_spread:.3f} > 2 SE = {2 * cfg_se:.3f})"
    else:
        verdict = (f"NOT RESOLVABLE (spread {cfg_spread:.3f} <= 2 SE = {2 * cfg_se:.3f}) "
                   "— take the most regularized config")
    return pd.DataFrame([{
        "n_configs": n_cfg, "runs_per_config": round(n_per_cfg, 1),
        "fold_sd": round(fold_sd, 3) if np.isfinite(fold_sd) else np.nan,
        "seed_sd": round(seed_sd, 3),
        "config_sd": round(cfg_sd, 3) if np.isfinite(cfg_sd) else np.nan,
        "config_spread": round(cfg_spread, 3) if np.isfinite(cfg_spread) else np.nan,
        "config_mean_se": round(cfg_se, 3),
        "verdict": verdict}])


def table_paired_configs(df: pd.DataFrame) -> pd.DataFrame:
    """Every configuration against the best one, PAIRED on (fold, seed).

    Comparing configuration means across folds throws the pairing away, and fold
    difficulty dominates the variance — so an unpaired comparison can call a real
    difference a tie. Every configuration here saw the identical (fold, seed)
    cells, so differencing within a cell cancels fold difficulty exactly and
    leaves only the configuration effect.

    `delta` is (this config - reference); POSITIVE means worse than the
    reference. `t` is the paired t-statistic; |t| > ~2 with n cells is a
    difference the sweep can actually see. Reported as a descriptive statistic,
    NOT a hypothesis test — no multiplicity correction is applied and the cells
    are not independent across folds.
    """
    if df.empty or df["config"].nunique() < 2:
        return pd.DataFrame()
    m = "smoothed_best_set_mae"
    wide = df.pivot_table(index=["val_pids", "seed"], columns="config", values=m)
    ref = wide.mean().idxmin()
    rows = []
    for cfg in wide.columns:
        if cfg == ref:
            continue
        d = (wide[cfg] - wide[ref]).dropna()
        if d.empty:
            continue
        sd = float(d.std(ddof=1)) if len(d) > 1 else np.nan
        se = sd / np.sqrt(len(d)) if np.isfinite(sd) and len(d) else np.nan
        rows.append({"config": cfg, "vs_reference": ref, "n_pairs": int(len(d)),
                     "delta": float(d.mean()), "delta_sd": sd, "delta_se": se,
                     "t": float(d.mean() / se) if se and np.isfinite(se) and se > 0 else np.nan,
                     "worse_than_ref": bool(d.mean() > 0)})
    out = pd.DataFrame(rows)
    return _fmt(out.sort_values("delta").reset_index(drop=True),
                ["delta", "delta_sd", "delta_se", "t"]) if not out.empty else out

In [ ]:
# ------------------------------------------------------------------- plots

def plot_val_curves(name: str, title: str):
    """Validation curves vs epoch — set-MAE (top row) and set-accuracy (bottom).

    **Left column** plots each metric MINUS that fold's own constant floor,
    averaged over folds and seeds, one line per configuration. Plotting raw
    values would be misleading: folds differ in difficulty by ~0.9 MAE, so an
    average across them mixes levels and the shape would be dominated by which
    folds a configuration happened to run. On these axes **y = 0 is the constant
    baseline**.

    **The two rows have OPPOSITE good directions** and the axis labels say so:
    for set-MAE lower is better, so below zero beats the floor; for
    set-accuracy higher is better, so above zero beats it. The floors are also
    two different predictors — the MAE-optimal constant and the
    accuracy-optimal constant are usually different LoAs (`const_loa_mae` vs
    `const_loa_acc`), which is why each row uses its own.

    **Right column** takes the best configuration and shows one raw line per
    fold, with that fold's floor as a matching dotted line. The spread between
    folds is the driver heterogeneity everything else here is about. Both rows
    use the SAME configuration — chosen on MAE, the design's selection metric —
    so the two views describe one model, not two.

    The dashed grey line is the untrained init; the vertical line is
    `--min-select-epoch`. A curve starting past the floor and moving away from
    it is a model whose held-out performance is degraded by training; a genuine
    interior optimum is one worth selecting an epoch from.
    """
    curves = load_curves(name)
    if curves.empty:
        print("(no per-epoch curves in runs/)")
        return
    need = {"const_set_mae", "const_set_acc", "set_acc"}
    if not need.issubset(curves.columns):
        print(f"(curves lack {sorted(need - set(curves.columns))} — re-run the sweep)")
        return
    meta = curves["tag"].map(_config_from_tag)
    curves = curves.assign(config=[m[0] for m in meta], val_pids=[m[3] for m in meta])
    curves["d_mae"] = curves["set_mae"] - curves["const_set_mae"]
    curves["d_acc"] = curves["set_acc"] - curves["const_set_acc"]

    # One configuration drives both right-hand panels: picked on MAE, because
    # that is what the sweep selects on. Accuracy is reported, never selected on.
    best = curves.groupby("config")["d_mae"].mean().idxmin()

    fig, axes = plt.subplots(2, 2, figsize=(13, 8.4))
    cmap = plt.get_cmap("tab10")

    rows = [
        # (row, delta col, raw col, floor col, ylabel, init col, better)
        (0, "d_mae", "set_mae", "const_set_mae",
         "set-MAE minus fold's floor", "init_set_mae", "lower is better"),
        (1, "d_acc", "set_acc", "const_set_acc",
         "set-accuracy minus fold's floor", "init_set_acc", "higher is better"),
    ]
    for r, dcol, raw, floor, ylab, initcol, better in rows:
        axL, axR = axes[r][0], axes[r][1]

        for cfg, c in curves.groupby("config"):
            s = c.groupby("epoch")[dcol].mean()
            axL.plot(s.index, s.values, lw=1.6, label=cfg)
        axL.axhline(0, color="crimson", lw=1.4, label="constant floor (y=0)")
        if initcol in curves.columns and curves[initcol].notna().any():
            init_d = float((curves[initcol] - curves[floor]).mean())
            axL.axhline(init_d, color="0.45", ls="--", lw=1.2,
                        label=f"untrained init ({init_d:+.2f})")
        axL.axvline(MIN_SELECT_EPOCH, color="0.7", ls=":", lw=1.2)
        axL.set_xlabel("epoch")
        axL.set_ylabel(f"{ylab}\n({better})", fontsize=9)
        axL.set_title(f"vs constant floor — {better} (mean over folds & seeds)", fontsize=10)
        axL.grid(alpha=0.3)
        axL.legend(fontsize=7)

        sub = curves[curves["config"] == best]
        for i, (fold, c) in enumerate(sub.groupby("val_pids")):
            col = cmap(i % 10)
            s = c.groupby("epoch")[raw].mean()
            axR.plot(s.index, s.values, lw=1.5, color=col, label=f"val={fold}")
            axR.axhline(float(c[floor].iloc[0]), color=col, ls=":", lw=1.0)
        axR.axvline(MIN_SELECT_EPOCH, color="0.7", ls=":", lw=1.2)
        axR.set_xlabel("epoch")
        axR.set_ylabel(f"{raw} (raw)\n({better})", fontsize=9)
        axR.set_title(f"per fold — {best}  (dotted = that fold's floor)", fontsize=10)
        axR.grid(alpha=0.3)
        axR.legend(fontsize=7)

    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    plt.show()


def plot_histograms(df: pd.DataFrame, title: str, ncols: int = 3):
    """One panel per configuration x fold; three bars per LoA.

    * **pred (mean of seeds)** — how the selected model spread its predictions.
    * **val labels** — what the validation drivers marked; what it is scored on.
    * **train labels** — what the training drivers marked; what it is rewarded
      for reproducing.

    Normalised to SHARES: the totals are not comparable in counts (predictions
    are one per segment, labels one per MARK, and the training set has ~5x the
    segments of a fold's validation set).

    Predictions tracking **train** while **val** differs is a model that learned
    its training drivers and failed to transfer. Predictions piled on one LoA,
    matching neither, is collapse. Aggregate set-MAE cannot tell them apart.
    """
    if df.empty:
        print("(no data)")
        return
    pred_c = [f"pred_loa{c}" for c in LOA]
    lbl_c = [f"lbl_loa{c}" for c in LOA]
    trn_c = [f"trn_loa{c}" for c in LOA]
    if not all(c in df.columns for c in pred_c + lbl_c + trn_c):
        print("(missing histogram columns — re-run the sweep)")
        return
    combos = df.groupby(["config", "val_pids"], as_index=False).agg(
        **{c: (c, "mean") for c in pred_c},
        **{c: (c, "first") for c in lbl_c + trn_c},
        n_seeds=("seed", "size"))
    n = len(combos)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.3 * nrows), squeeze=False)
    width, x = 0.27, np.arange(len(LOA))
    for i, (_, r) in enumerate(combos.iterrows()):
        ax = axes[i // ncols][i % ncols]
        def share(cols):
            v = np.array([float(r[c]) for c in cols], dtype=float)
            t = v.sum()
            return v / t if t > 0 else v
        ax.bar(x - width, share(pred_c), width, color="#4C72B0",
               label=f"pred (mean of {int(r['n_seeds'])} seeds)")
        ax.bar(x, share(lbl_c), width, color="#DD8452", label="val labels")
        ax.bar(x + width, share(trn_c), width, color="#A0A0A0", label="train labels")
        ax.set_title(f"{r['config']}  |  val={r['val_pids']}", fontsize=9)
        ax.set_xticks(x); ax.set_xticklabels([f"LoA{c}" for c in LOA], fontsize=8)
        ax.set_ylabel("share", fontsize=8); ax.grid(axis="y", alpha=0.3)
        if i == 0:
            ax.legend(fontsize=7)
    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    plt.show()

In [ ]:
def report_stage(name: str, title: str):
    """Everything for one stage, identical order and format across all four."""
    df = load_stage(name)
    display(Markdown(f"### {title}"))
    if df.empty:
        print(f"(stage '{name}' has not run — no sweep_results.csv)")
        return
    w = df["window_seconds"].iloc[0] if "window_seconds" in df else "?"
    lo = df["loss"].iloc[0] if "loss" in df else "?"
    print(f"{len(df)} runs | loss={lo} | window={w}s | "
          f"{df['config'].nunique()} configuration(s) | "
          f"{df['val_pids'].nunique()} fold(s) | {df['seed'].nunique()} seed(s)")
    pr = per_run_curves(name)

    display(Markdown("**Table 1 — per configuration** (mean over folds and seeds)"))
    display(table_per_config(df))

    display(Markdown("**Table 2 — per configuration x validation fold** (mean over seeds)"))
    display(table_per_config_fold(df))

    display(Markdown(f"**Table 3 — epochs below the floor** "
                     f"(how often an epoch < {MIN_SELECT_EPOCH} would have won)"))
    t = table_early_wins(pr)
    display(t if not t.empty else "(no per-epoch curves in runs/)")

    display(Markdown("**Table 4 — untrained init vs selected model vs constant floor**"))
    t = table_init_vs_best(pr)
    display(t if not t.empty else "(curves predate the init columns — re-run to populate)")

    display(Markdown("**Table 5 — variance decomposition** (can this sweep rank configurations?)"))
    display(table_variance(df))

    display(Markdown("**Table 6 — paired configuration comparison** (fold difficulty cancelled)"))
    t = table_paired_configs(df)
    display(t if not t.empty else "(single configuration — nothing to pair)")

    plot_val_curves(name, f"{title} — validation curves")
    plot_histograms(df, f"{title} — prediction vs label distributions")

---
## 1. Full grid — soft-CORN, 10 s window

The primary run: 3 dropout x 2 lr x 6 folds x 5 seeds. Its winning `(dropout, lr)` is
what stages 3 and 4 inherit, so this is the only stage whose configuration ranking
feeds anything downstream — which is exactly why **Table 5** matters here: if the
ranking is not resolvable, the inherited winner is arbitrary.

In [ ]:
report_stage("corn_w10", "1. Full grid — soft-CORN, 10 s window")

---
## 2. Softmax / cross-entropy ablation — 10 s window

The ablation, not a candidate: the Laplace UQ layer refuses a non-CORN head, so CE
cannot be the deployed arm whatever it scores.

**Reading the CE-vs-CORN gap fairly.** The default decoder differs by head — argmax
(the mode) for softmax, the rank rule (the median) for CORN — and the median is optimal
for absolute error while the mode is optimal for accuracy. Comparing Table 1 across
sections 1 and 2 therefore measures *head and decoder together*. The cell after this one
holds the decoder fixed.

In [ ]:
report_stage("ce_w10", "2. Softmax / CE ablation — 10 s window")

In [ ]:
# CORN vs CE at a FIXED decoder — the controlled comparison.
rows = []
for name in ("corn_w10", "ce_w10"):
    d = load_stage(name)
    if d.empty or "mae_argmax" not in d:
        continue
    rows.append({"stage": name, "head": d["loss"].iloc[0] if "loss" in d else "?",
                 "n_runs": len(d),
                 "MAE @argmax (mode)": d["mae_argmax"].mean(),
                 "acc @argmax": d["acc_argmax"].mean(),
                 "MAE @median (rank rule)": d["mae_median"].mean(),
                 "acc @median": d["acc_median"].mean(),
                 "const floor": d["const_set_mae"].mean()})
if rows:
    t = pd.DataFrame(rows).round(3)
    display(Markdown("**Head vs decoder, disentangled** — compare *down* a column "
                     "(head effect at a fixed decoder), not across columns."))
    display(t)
    if len(t) == 2:
        for col in ("MAE @argmax (mode)", "MAE @median (rank rule)"):
            d = t[col].iloc[0] - t[col].iloc[1]
            better = t["head"].iloc[0] if d < 0 else t["head"].iloc[1]
            print(f"{col}: {better} better by {abs(d):.3f} MAE")
        print("\nIf both rows agree on which head wins, the advantage is the HEAD.")
        print("If they disagree, it was the decoder — and the design's metric is set-MAE.")
else:
    print("(need both corn_w10 and ce_w10 with per-decoder columns)")

---
## 3. 5 s window — soft-CORN, winning configuration

Single configuration inherited from stage 1, so Tables 1, 5 and 6 have one row or are
empty. The format is identical so it reads against sections 1 and 4.

The model sees only the last 5 s of each 20 s label window. The label cadence is
unchanged — one segment is still one label — so this varies *input context*, not the
amount of supervision. `context_length` becomes 50.

In [ ]:
report_stage("corn_w5", "3. 5 s window — soft-CORN")

---
## 4. 20 s window — soft-CORN, winning configuration

The full label window; `context_length` becomes 200. Same single-configuration format
as section 3.

In [ ]:
report_stage("corn_w20", "4. 20 s window — soft-CORN")

---
## Cross-stage roll-up

The window comparison (stages 1, 3, 4 — same loss and configuration, different input
context) and the head comparison (1 vs 2), in one table. The spread across windows is
checked against the seed noise before it is read as a result.

In [ ]:
rows = []
for name, title in STAGES:
    d = load_stage(name)
    if d.empty:
        continue
    best = table_per_config(d).iloc[0]          # sorted by set_mae
    pr = per_run_curves(name)
    init = float(pr["init_mae"].mean()) if not pr.empty and pr["init_mae"].notna().any() else np.nan
    rows.append({"stage": name,
                 "loss": d["loss"].iloc[0] if "loss" in d else "?",
                 "window_s": d["window_seconds"].iloc[0] if "window_seconds" in d else np.nan,
                 "best_config": best["config"], "n_runs": int(best["n_runs"]),
                 "init_mae": init,
                 "set_mae": best["set_mae"], "mae_se": best["mae_se"],
                 "set_acc": best["set_acc"], "const_mae": best["const_mae"],
                 "vs_const": best["vs_const"], "E_star": best["E_star"]})
if rows:
    roll = pd.DataFrame(rows).round(3)
    display(roll)
    corn = roll[roll["loss"] == "corn"]
    if len(corn) > 1:
        b = corn.loc[corn["set_mae"].idxmin()]
        spread = float(corn["set_mae"].max() - corn["set_mae"].min())
        se = float(corn["mae_se"].mean())
        print(f"\nBest CORN window: {b['window_s']:g}s (set-MAE {b['set_mae']:.3f})")
        print(f"Spread across windows: {spread:.3f} vs mean SE {se:.3f} -> "
              f"{'MEANINGFUL' if spread > 2 * se else 'WITHIN NOISE, do not over-read'}")
    if (roll["vs_const"] > 0).all():
        print("\nNOTE: no stage beats its constant floor. The window and head comparisons "
              "are then between models that all lose to 'always predict one level'. That "
              "is a property of the task on this cohort (driver identity is worth ~4x more "
              "MAE than the task), not necessarily of the runs — but it means the "
              "population model should be read as an INITIALIZATION for personalization, "
              "not as a result in itself. Check Table 4 in each section: if the models do "
              "beat their untrained init, they ARE learning, just not something that "
              "transfers across drivers.")
else:
    print("(no stages have run yet)")